# Part 3a: Process-based model — calibration evaluation: Moselle

## Introduction
This notebook runs the already-calibrated SuperflexPy model in forward mode over the Moselle calibration period (1991–2001) and saves the simulated streamflow as NetCDF files per group. The calibration itself (SCE-UA optimisation, ~10,000 model evaluations per group per experiment) was performed on a computing cluster using the scripts in `code/00_cluster/`; this notebook only loads the best-parameter CSVs and evaluates the model forward.

Alongside the other notebooks, it covers all the analysis performed in: "Assessing the Impact of Geological Map Detail on Process-Based and Data-Driven Hydrological Models" by do Nascimento et al. (2026).

Author: Thiago Nascimento (thiago.nascimento@eawag.ch)

## Requirements

**From EStreams dataset (v.1.4)** — https://doi.org/10.5281/zenodo.17598150 (Last access: 11 February 2025)

- `streamflow_gauges/estreams_gauging_stations.csv` — gauge metadata

**From this repository (`../data/`):**

- `estreams_attributes_filtered_quality_geology_v01.csv` — catchment attributes including global and continental permeability fractions (do Nascimento et al., 2025a; https://github.com/thiagovmdon/LSH-quality_geology)
- `estreams_geology_moselle_regional_attributes.csv` — regional-scale lithological fractions for 108 Moselle catchments (https://doi.org/10.5281/zenodo.18392387)
- `network_estreams_moselle_108_gauges.csv` — 108 Moselle study gauges and group assignments (produced by Part-1a Moselle)
- `models/input/subset_2001_2015/inputs.npy` — forcing arrays per group (produced by Part-B PB)
- `models/input/subset_2001_2015/observations.npy` — observed streamflow arrays
- `models/input/subset_2001_2015/perm_areas.npy` — regional HRU weights
- `models/input/subset_2001_2015/perm_areasglobal.npy` — global HRU weights
- `models/input/subset_2001_2015/perm_areascontinental.npy` — continental HRU weights
- `models/input/subset_2001_2015/quality_masks.npy`, `areas.npy`, `rootdepth_mean.npy`, `waterdeficit_mean.npy`

**Note:** The actual calibration (SCE-UA via SPOTPY) is run on a computing cluster using the scripts in `code/00_cluster/`. This notebook runs the forward model with the already-calibrated parameters and saves evaluation outputs. The cluster scripts produce `../results/groups/moselle_best_params_*.csv` files that are loaded here.

**Produced by this notebook:**

- `../results/sim/sim_moselle/calibration/simu_cal_Group_*.nc` — simulated streamflow for the calibration period, one NetCDF per group (used in Part-E)


# Import the modules

In [1]:
import pandas as pd
import datetime as datetime
import matplotlib.pyplot as plt
import numpy as np
import spotpy
import time
import os
import tqdm as tqdm
import hydroanalysis
from utils.functions import find_max_unique_rows
from utils.functions import find_iterative_immediate_downstream
import geopandas as gpd
#warnings.filterwarnings("ignore")

## Set the path to the data

In [2]:
# Path to where the EStreams dataset is stored
# Eawag
path_estreams = r'/Users/nascimth/Documents/data/EStreams'

## Mac
#path_estreams = r'/Users/thiagomedeirosdonascimento/Downloads/Python 2/Scripts/estreams_part_b/data/EStreams'

path_data = r"/Users/nascimth/Documents/data"

## Read the files

In [3]:
# Read the dataset network
network_estreams = pd.read_csv(path_estreams+'/streamflow_gauges/estreams_gauging_stations.csv', encoding='utf-8')
network_estreams.set_index("basin_id", inplace = True)

# Convert 'date_column' and 'time_column' to datetime
network_estreams['start_date'] = pd.to_datetime(network_estreams['start_date'])
network_estreams['end_date'] = pd.to_datetime(network_estreams['end_date'])

# Convert to list both the nested_catchments and the duplicated_suspect columns
network_estreams['nested_catchments'] = network_estreams['nested_catchments'].apply(lambda x: x.strip("[]").replace("'", "").split(", "))

# Remove the brackets and handle NaN values
network_estreams['duplicated_suspect'] = network_estreams['duplicated_suspect'].apply(
    lambda x: x.strip("[]").replace("'", "").split(", ") if isinstance(x, str) else x)

# Set the nested catchments as a dataframe
nested_catchments = pd.DataFrame(network_estreams['nested_catchments'])

# Now we add the outlet to the list (IF it was not before):
# Ensure that the basin_id is in the nested_catchments
for basin_id in nested_catchments.index:
    if basin_id not in nested_catchments.at[basin_id, 'nested_catchments']:
        nested_catchments.at[basin_id, 'nested_catchments'].append(basin_id)



# Attributes already filtered previously:
#estreams_attributes = pd.read_csv('data/exploration/estreams_attributes_filtered_moselle_sm_su_tog.csv', encoding='utf-8')
estreams_attributes = pd.read_csv('../data/estreams_attributes_filtered_quality_geology_v01.csv', encoding='utf-8')

estreams_attributes.set_index("basin_id", inplace = True)

# Convert to list both the nested_catchments and the duplicated_suspect columns
estreams_attributes['nested_catchments'] = estreams_attributes['nested_catchments'].apply(lambda x: x.strip("[]").replace("'", "").split(", "))

# Remove the brackets and handle NaN values
estreams_attributes['duplicated_suspect'] = estreams_attributes['duplicated_suspect'].apply(
    lambda x: x.strip("[]").replace("'", "").split(", ") if isinstance(x, str) else x)

estreams_attributes.sort_index(inplace = True) 

In [4]:
# Geological attributes (regional scale)
geology_regional_31_classes_moselle = pd.read_csv("../data/estreams_geology_moselle_regional_attributes.csv", encoding='utf-8')

geology_regional_31_classes_moselle.set_index("basin_id", inplace = True)

# Create a dictionary to map permeability classes to corresponding columns
permeability_columns = {
    "high": ["lit_fra_Alluvium", 'lit_fra_Coal', 'lit_fra_Conglomerate', 'lit_fra_Gravel and sand',
             'lit_fra_Sand', 'lit_fra_Sand and gravel', 'lit_fra_Sandstone and conglomerate', 'lit_fra_Sandstone'
        ],
    
    "medium": ['lit_fra_Limestone', 'lit_fra_Sandstone and marl', 'lit_fra_Sandstone and schist',
              'lit_fra_Sandstone, conglomerate and marl',

              'lit_fra_Arkose', 'lit_fra_Dolomite rock', 'lit_fra_Limestone and marl', 'lit_fra_Marl', 
             'lit_fra_Marl and dolomite', 'lit_fra_Marl and limestone', 'lit_fra_Marl and sandstone',
               'lit_fra_Sandstone and siltstone', 'lit_fra_Sandstone, siltstone and schist', 
              'lit_fra_Schist and sandstone', 'lit_fra_Silt',  'lit_fra_Silt and schist', 'lit_fra_Siltstone, sandstone and schist'
              
             ],
    
    "low": ['lit_fra_Cristallin basement', 'lit_fra_Plutonic rock',  'lit_fra_Quarzite',
                    'lit_fra_Schist','lit_fra_Volcanic rock' 
                   ]
}

# Iterate over the permeability columns and calculate the area for each class
for permeability_class, columns in permeability_columns.items():
    geology_regional_31_classes_moselle[f'area_perm_{permeability_class}'] = geology_regional_31_classes_moselle[columns].sum(axis=1)

# Drop unnecessary columns
geology_regional_31_classes_moselle = geology_regional_31_classes_moselle[["area_perm_high", "area_perm_medium", "area_perm_low"]]

# Rename the columns
geology_regional_31_classes_moselle.columns = ["perm_high_regi", "perm_medium_regi", "perm_low_regi"]

# Display the updated DataFrame
geology_regional_31_classes_moselle

geology_regional_31_classes_moselle["baseflow_index"] = estreams_attributes["baseflow_index"]
geology_regional_31_classes_moselle.corr(method="pearson")

# Concatenation
estreams_attributes[["perm_high_regi", "perm_medium_regi", "perm_low_regi"]] = geology_regional_31_classes_moselle[["perm_high_regi", "perm_medium_regi", "perm_low_regi"]]

# Adjust the three categories for also global dataset
estreams_attributes["perm_high_glob2"] = estreams_attributes["perm_high_glob"]
estreams_attributes["perm_medium_glob2"] = estreams_attributes["perm_medium_glob"] + estreams_attributes["perm_low_glob"]
estreams_attributes["perm_low_glob2"] = estreams_attributes["perm_verylow_glob"]

###########################################################################################################################
# Adjust the columns of the dataset:
for basin_id in estreams_attributes.index.tolist():

    # Extract and divide by 100
    v1 = estreams_attributes.loc[basin_id, "perm_high_regi"] / 100
    v2 = estreams_attributes.loc[basin_id, "perm_medium_regi"] / 100
    v3 = estreams_attributes.loc[basin_id, "perm_low_regi"] / 100

    # Round all values to one decimal place
    v1 = round(v1, 2)
    v2 = round(v2, 2)
    v3 = round(v3, 2)

    # Ensure the sum is exactly 1 by adjusting the largest value
    diff = 1 - (v1 + v2 + v3)

    if diff != 0:
        # Adjust the value that was the largest before rounding
        if max(v1, v2, v3) == v1:
            v1 += diff
        elif max(v1, v2, v3) == v2:
            v2 += diff
        else:
            v3 += diff

    # Assign back
    estreams_attributes.loc[basin_id, "perm_high_regi"] = v1 * 100
    estreams_attributes.loc[basin_id, "perm_medium_regi"] = v2 * 100
    estreams_attributes.loc[basin_id, "perm_low_regi"] = v3 * 100


for basin_id in estreams_attributes.index.tolist():

    # Extract and divide by 100
    v1 = estreams_attributes.loc[basin_id, "perm_high_glob2"] / 100
    v2 = estreams_attributes.loc[basin_id, "perm_medium_glob2"] / 100
    v3 = estreams_attributes.loc[basin_id, "perm_low_glob2"] / 100

    # Round all values to one decimal place
    v1 = round(v1, 2)
    v2 = round(v2, 2)
    v3 = round(v3, 2)

    # Ensure the sum is exactly 1 by adjusting the largest value
    diff = 1 - (v1 + v2 + v3)

    if diff != 0:
        # Adjust the value that was the largest before rounding
        if max(v1, v2, v3) == v1:
            v1 += diff
        elif max(v1, v2, v3) == v2:
            v2 += diff
        else:
            v3 += diff

    # Assign back
    estreams_attributes.loc[basin_id, "perm_high_glob2"] = v1 * 100
    estreams_attributes.loc[basin_id, "perm_medium_glob2"] = v2 * 100
    estreams_attributes.loc[basin_id, "perm_low_glob2"] = v3 * 100

In [5]:
# Define the functions
def obj_fun_nsee(observations, simulation, expo=0.5):
    """
    Calculate the Normalized Squared Error Efficiency (NSEE) while ensuring that
    NaNs in simulation are NOT masked (only NaNs in observations are masked).

    Parameters:
        observations (array-like): Observed values (with fixed NaNs).
        simulation (array-like): Simulated values (can contain NaNs).
        expo (float, optional): Exponent applied to observations and simulations. Default is 1.0.

    Returns:
        float: NSEE score (higher values indicate worse performance).
    """
    observations = np.asarray(observations)
    simulation = np.asarray(simulation)

    # Mask only NaNs in observations
    mask = ~np.isnan(observations)
    obs = observations[mask]
    sim = simulation[mask]  # Keep all simulated values, even NaNs

    # If simulation contains NaNs after masking observations, return penalty
    if np.isnan(sim).any():
        return 10.0  # Large penalty if NaNs appear in the simulation

    metric = np.sum((sim**expo - obs**expo)**2) / np.sum((obs**expo - np.mean(obs**expo))**2)
    
    return float(metric)


def obj_fun_kge(observations, simulation):
    """
    Calculate the KGE-2012 objective function, ensuring that NaNs in simulation are NOT masked.
    
    Parameters:
        observations (array-like): Observed values (with fixed NaNs).
        simulation (array-like): Simulated values (can contain NaNs).

    Returns:
        float: KGE-2012 score (higher values indicate worse performance).
    """
    observations = np.asarray(observations)
    simulation = np.asarray(simulation)

    # Mask only NaNs in observations
    mask = ~np.isnan(observations)
    obs = observations[mask]
    sim = simulation[mask]  # Keep all simulated values, even NaNs

    # Check if there are NaNs in the simulation after masking obs
    if np.isnan(sim).any():
        return 10.0  # Large penalty if the simulation contains NaNs
    
    obs_mean = np.mean(obs)
    sim_mean = np.mean(sim)

    r = np.corrcoef(obs, sim)[0, 1]
    alpha = np.std(sim) / np.std(obs)
    beta = sim_mean / obs_mean

    kge = np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)  # KGE-2012

    return float(kge)

In [6]:
# First we define the outlet of the Moselle to be used
outlets = ["DEBU1959"]
nested_cats_df = nested_catchments.loc[outlets, :]

# Now we save our dataframes in a dictionary of dataframes. One dataframe for each watershed. 

nested_cats_filtered = find_max_unique_rows(nested_cats_df)                                  # Filter only the catchemnts using the function stated before
nested_cats_filtered_df = nested_catchments.loc[nested_cats_filtered, :]                     # Here we filter the catchemnts for the list (again, after we apply our function):

# Store the variables for the selected catchments in a list of dataframes now for only the ones above 20 cats:
estreams_attributes_dfs = {}
for catchment in tqdm.tqdm(nested_cats_filtered):
    # Retrieve the nested list of catchments for the current catchment
    nested_clip = nested_cats_filtered_df.loc[catchment, 'nested_catchments']
    
    # Filter values to include only those that exist in the index of estreams_attributes
    nested_clip = [value for value in nested_clip if value in estreams_attributes.index]
    
    # Filter the estreams_attributes DataFrame based on the filtered nested_clip
    cat_clip = estreams_attributes.loc[nested_clip, :]
    
    # Store the resulting DataFrame in the dictionary
    estreams_attributes_dfs[catchment] = cat_clip

# Here we can save the length of each watershed (number of nested catchemnts)
catchment_lens = pd.DataFrame(index = estreams_attributes_dfs.keys())
for catchment, data in estreams_attributes_dfs.items():
    catchment_lens.loc[catchment, "len"] = len(data)

# Now we can filter it properly:
nested_cats_filtered_abovevalue = catchment_lens[catchment_lens.len >= 10]

# # Here we filter the catchemnts for the list (again, after we apply our function):
nested_cats_filtered_abovevalue_df = nested_catchments.loc[nested_cats_filtered_abovevalue.index, :]

# Store the variables for the selected catchments in a list of dataframes now for only the ones above 20 cats:
estreams_attributes_dfs = {}

for catchment in tqdm.tqdm(nested_cats_filtered_abovevalue_df.index):
    # Retrieve the nested list of catchments for the current catchment
    nested_clip = nested_cats_filtered_abovevalue_df.loc[catchment, 'nested_catchments']
    
    # Filter values to include only those that exist in the index of estreams_attributes
    nested_clip = [value for value in nested_clip if value in estreams_attributes.index]
    
    # Filter the estreams_attributes DataFrame based on the filtered nested_clip
    cat_clip = estreams_attributes.loc[nested_clip, :]
    
    # Store the resulting DataFrame in the dictionary
    estreams_attributes_dfs[catchment] = cat_clip

# Adjust and clip it:
estreams_attributes_clipped = estreams_attributes_dfs["DEBU1959"]

# Convert 'date_column' and 'time_column' to datetime
estreams_attributes_clipped['start_date'] = pd.to_datetime(estreams_attributes_clipped['start_date'])
estreams_attributes_clipped['end_date'] = pd.to_datetime(estreams_attributes_clipped['end_date'])


#estreams_attributes_clipped_filters = estreams_attributes_clipped[estreams_attributes_clipped.end_date >= "2010"]
#estreams_attributes_clipped_filters = estreams_attributes_clipped_filters[estreams_attributes_clipped_filters.start_date <= "2002"]

# Here we retrieve the conectivity (from EStreams computation)
# Load the nested catchments CSV file
df = pd.read_excel("../data/nested_catchments.xlsx")

# Rename columns for clarity
df = df.rename(columns={df.columns[1]: "basin_id", df.columns[2]: "connected_basin_id"})
df = df.drop(columns=[df.columns[0]])  # Drop the unnamed index column

100%|██████████| 1/1 [00:00<00:00, 422.13it/s]


In [7]:
# Read the dataset network
estreams_attributes_clipped_filters = pd.read_csv(R'../data/network_estreams_moselle_108_gauges.csv', encoding='utf-8')
estreams_attributes_clipped_filters.set_index("basin_id", inplace = True)
estreams_attributes_clipped_filters

,Unnamed: 0,gauge_id,gauge_name,gauge_country,gauge_provider,river,lon_snap,lat_snap,lon,lat,...,irri_1990,irri_2005,stations_num_p_mean,perm_high_regi,perm_medium_regi,perm_low_regi,perm_high_glob2,perm_medium_glob2,perm_low_glob2,group
basin_id,,,,,,,,,,,,,,,,,,,,,
LU000018,0,5,Schoenfels,LU,LU_CONTACTFORM,Mamer,6.100795,49.723112,6.100795,49.723112,...,0.015,0.015,17.0,39.0,61.0,0.0,0.0,100.0,0.0,Group_1
LU000010,1,6,Hunnebuer,LU,LU_CONTACTFORM,Eisch,6.079524,49.729184,6.079524,49.729184,...,0.026,0.026,16.0,42.0,58.0,0.0,1.0,99.0,0.0,Group_1
LU000001,2,17,Bigonville,LU,LU_CONTACTFORM,Sure,5.801399,49.869821,5.801399,49.869821,...,0.000,0.000,9.0,1.0,0.0,99.0,100.0,0.0,0.0,Group_1
DERP2028,3,2674030900,Eisenschmitt,DE,DE_RP,Salm,6.718000,50.048000,6.718000,50.048000,...,0.000,0.000,10.0,80.0,7.0,13.0,79.0,20.0,1.0,Group_1
FR000183,4,A900105050,A9001050,FR,FR_EAUFRANCE,La Sarre à Laneuveville-lès-Lorquin,7.008689,48.654579,7.008689,48.654579,...,0.000,0.000,4.0,66.0,29.0,5.0,64.0,30.0,6.0,Group_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR003271,107,A782101001,La Seille Ã Nomeny,FR,FR_EAUFRANCE,La Seille à Nomeny,6.227788,48.888271,6.227788,48.888271,...,0.429,0.436,5.0,8.0,92.0,0.0,0.0,100.0,0.0,Group_7
FR003301,108,A930108040,La Sarre Ã Wittring,FR,FR_EAUFRANCE,La Sarre à Wittring,7.150066,49.053225,7.150066,49.053225,...,0.436,2.205,16.0,17.0,83.0,0.0,15.0,85.0,0.0,Group_7
DERP2003,109,2620050500,Bollendorf,DE,DE_RP,Sauer,6.359000,49.851000,6.359000,49.851000,...,1.627,4.160,65.0,17.0,30.0,53.0,50.0,50.0,0.0,Group_7


In [8]:
# Python implementation
from superflexpy.framework.unit import Unit
from superflexpy.framework.node import Node
from superflexpy.framework.network import Network

from superflexpy.implementation.elements.hbv import UnsaturatedReservoir, PowerReservoir

from superflexpy.implementation.numerical_approximators.implicit_euler import ImplicitEulerPython
from superflexpy.implementation.root_finders.pegasus import PegasusPython

# Numba implementation:
from superflexpy.implementation.root_finders.pegasus import PegasusNumba
from superflexpy.implementation.numerical_approximators.implicit_euler import ImplicitEulerNumba

from superflexpy.implementation.elements.hbv import PowerReservoir
from superflexpy.framework.unit import Unit
from superflexpy.implementation.elements.thur_model_hess import SnowReservoir, UnsaturatedReservoir, PowerReservoir, HalfTriangularLag

from superflexpy.implementation.elements.structure_elements import Transparent, Junction, Splitter
from superflexpy.framework.element import ParameterizedElement

In [9]:
root_finder = PegasusNumba()
num_app = ImplicitEulerNumba(root_finder=root_finder)

class ParameterizedSingleFluxSplitter(ParameterizedElement):
    _num_downstream = 2
    _num_upstream = 1
    
    def set_input(self, input):

        self.input = {'Q_in': input[0]}

    def get_output(self, solve=True):

        split_par = self._parameters[self._prefix_parameters + 'splitpar']

        output1 = [self.input['Q_in'] * split_par]
        output2 = [self.input['Q_in'] * (1 - split_par)]
        
        return [output1, output2]   
    
    
lower_splitter = ParameterizedSingleFluxSplitter(
    parameters={'splitpar': 0.5},
    id='lowersplitter'
)

lower_splitter_medium = ParameterizedSingleFluxSplitter(
    parameters={'splitpar': 0.6},
    id='lowersplitter'
)

lower_splitter_high = ParameterizedSingleFluxSplitter(
    parameters={'splitpar': 0.7},
    id='lowersplitter'
)

# Fluxes in the order P, T, PET
upper_splitter = Splitter(
    direction=[
        [0, 1, None],    # P and T go to the snow reservoir
        [2, None, None]  # PET goes to the transparent element
    ],
    weight=[
        [1.0, 1.0, 0.0],
        [0.0, 0.0, 1.0]
    ],
    id='upper-splitter'
)

snow = SnowReservoir(
    parameters={'t0': 0.0, 'k': 0.01, 'm': 2.0},
    states={'S0': 0.0},
    approximation=num_app,
    id='snow'
)

upper_transparent = Transparent(
    id='upper-transparent'
)

upper_junction = Junction(
    direction=[
        [0, None],
        [None, 0]
    ],
    id='upper-junction'
)


unsaturated = UnsaturatedReservoir(
    parameters={'Smax': 150.0, 'Ce': 1.0, 'm': 0.01, 'beta': 2.0},
    states={'S0': 10.0},
    approximation=num_app,
    id='unsaturated'
)

fast = PowerReservoir(
    parameters={'k': 0.01, 'alpha': 2.0},
    states={'S0': 0.0},
    approximation=num_app,
    id='fast'
)

slow = PowerReservoir(
    parameters={'k': 1e-4, 'alpha': 1.0},
    states={'S0': 0.0},
    approximation=num_app,
    id='slow'
)

slowhigh = PowerReservoir(
    parameters={'k': 1e-4, 'alpha': 2.0},
    states={'S0': 0.0},
    approximation=num_app,
    id='slowhigh'
)


lower_junction = Junction(
    direction=[
        [0, 0]
    ],
    id='lower-junction'
)

lag_fun = HalfTriangularLag(
    parameters={'lag-time': 4.0},
    states={'lag': None},
    id='lag-fun'
)

lower_transparent = Transparent(
    id='lower-transparent'
)

lower_transparent2 = Transparent(
    id='lower-transparent2'
)

general = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
    id='general'
)

low = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [fast],
    ],
    id='low'
)

high = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [slowhigh],
    ],
    id='high'
)

In [10]:
import os
import glob

# Dictionary to store all parameter dicts
all_param_dicts = {}

# Loop through all CSVs in the current directory
for filepath in glob.glob("../results/groups/*moselle*.csv"):
    file_key = os.path.splitext(os.path.basename(filepath))[0]  # Strip .csv
    
    param_dict = {}

    # Read file and parse lines
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith(","):  # Skip empty or malformed lines
                continue
            parts = line.split(",")
            if len(parts) == 2:
                key, value = parts
                try:
                    param_dict[key] = float(value)
                except ValueError:
                    pass  # Skip lines where value is not a float
            else:
                pass  # Skip malformed lines

    # Store the parsed dictionary
    all_param_dicts[file_key] = param_dict


In [11]:
catchments_ids = estreams_attributes_clipped_filters.index.tolist()

def calculate_hydro_year(date, first_month=10):
    """
    This function calculates the hydrological year from a date. The
    hydrological year starts on the month defined by the parameter first_month.

    Parameters
    ----------
    date : pandas.core.indexes.datetimes.DatetimeIndex
        Date series
    first_month : int
        Number of the first month of the hydrological year

    Returns
    -------
    numpy.ndarray
        Hydrological year time series
    """

    hydrological_year = date.year.values.copy()
    hydrological_year[date.month >= first_month] += 1

    return hydrological_year

def run_model_superflexpy(catchments_ids, best_params_dict_model, perm_areas_model):
    # Run the iterative function
    iterative_immediate_downstream = find_iterative_immediate_downstream(df, catchments_ids)

    # Convert results to a DataFrame for display
    iterative_downstream_df = pd.DataFrame(iterative_immediate_downstream.items(), 
                                        columns=['basin_id', 'immediate_downstream_basin'])


    # Assuming the DataFrame has columns 'basin_id' and 'downstream_id'
    topology_list = {basin: None for basin in catchments_ids}  # Default to None

    # Filter DataFrame for relevant basin_ids and update topology
    for _, row in iterative_downstream_df.iterrows():
        if row['basin_id'] in topology_list:
            topology_list[row['basin_id']] = row['immediate_downstream_basin']

    # Generate Nodes dynamically and assign them as global variables
    catchments = [] # Dictionary to store nodes
    
    general = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
    id='general')

    low = Unit(
        layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
        id='low')

    high = Unit(
        layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
        id='high')

    for cat_id in catchments_ids:
        node = Node(
            units=[high, general, low],  # Use unit from dictionary or default
            weights=perm_areas_model[cat_id],
            area=areas.get(cat_id),  # Use predefined area or default
            id=cat_id
        )
        catchments.append(node)  # Store in the list

        # Assign the node as a global variable
        globals()[cat_id] = node


    # Ensure topology only includes nodes that exist in `catchments_ids`
    topology = {
        cat_id: upstream if upstream in catchments_ids else None
        for cat_id, upstream in topology_list.items() if cat_id in catchments_ids
    }

    # Create the Network
    model = Network(
        nodes=catchments,  # Pass list of Node objects
        topology=topology  
    )

    model.reset_states()

    # Set inputs for each node using the manually defined dictionary
    for cat in catchments:
        cat.set_input(inputs[cat.id])  # Correct way to set inputs

    model.set_timestep(1.0)
    model.set_parameters(best_params_dict_model)

    output = model.get_output()

    return output

def run_model_superflexpy_nogeo(catchments_ids, best_params_dict_model, perm_areas_model):
    # Run the iterative function
    iterative_immediate_downstream = find_iterative_immediate_downstream(df, catchments_ids)

    # Convert results to a DataFrame for display
    iterative_downstream_df = pd.DataFrame(iterative_immediate_downstream.items(), 
                                        columns=['basin_id', 'immediate_downstream_basin'])


    # Assuming the DataFrame has columns 'basin_id' and 'downstream_id'
    topology_list = {basin: None for basin in catchments_ids}  # Default to None

    # Filter DataFrame for relevant basin_ids and update topology
    for _, row in iterative_downstream_df.iterrows():
        if row['basin_id'] in topology_list:
            topology_list[row['basin_id']] = row['immediate_downstream_basin']

    # Generate Nodes dynamically and assign them as global variables
    catchments = [] # Dictionary to store nodes
    
    general = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
    id='general')

    for cat_id in catchments_ids:
        node = Node(
            units=[general],  # Use unit from dictionary or default
            weights=[1.0],
            area=areas.get(cat_id),  # Use predefined area or default
            id=cat_id
        )
        catchments.append(node)  # Store in the list

        # Assign the node as a global variable
        globals()[cat_id] = node


    # Ensure topology only includes nodes that exist in `catchments_ids`
    topology = {
        cat_id: upstream if upstream in catchments_ids else None
        for cat_id, upstream in topology_list.items() if cat_id in catchments_ids
    }

    # Create the Network
    model = Network(
        nodes=catchments,  # Pass list of Node objects
        topology=topology  
    )

    model.reset_states()

    # Set inputs for each node using the manually defined dictionary
    for cat in catchments:
        cat.set_input(inputs[cat.id])  # Correct way to set inputs

    model.set_timestep(1.0)
    model.set_parameters(best_params_dict_model)

    output = model.get_output()

    return output

def run_model_superflexpy_random(catchments_ids, best_params_dict_model, perm_areas_model):
    # Run the iterative function
    iterative_immediate_downstream = find_iterative_immediate_downstream(df, catchments_ids)

    # Convert results to a DataFrame for display
    iterative_downstream_df = pd.DataFrame(iterative_immediate_downstream.items(), 
                                        columns=['basin_id', 'immediate_downstream_basin'])


    # Assuming the DataFrame has columns 'basin_id' and 'downstream_id'
    topology_list = {basin: None for basin in catchments_ids}  # Default to None

    # Filter DataFrame for relevant basin_ids and update topology
    for _, row in iterative_downstream_df.iterrows():
        if row['basin_id'] in topology_list:
            topology_list[row['basin_id']] = row['immediate_downstream_basin']

    # Generate Nodes dynamically and assign them as global variables
    catchments = [] # Dictionary to store nodes
    
    general = Unit(
    layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
    id='general')

    low = Unit(
        layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
        id='low')

    high = Unit(
        layers=[
        [upper_splitter],
        [snow, upper_transparent],
        [upper_junction],
        [unsaturated],
        [lower_splitter],
        [slow, lag_fun],
        [lower_transparent, fast],
        [lower_junction],
    ],
        id='high')

    for cat_id in catchments_ids:
        node = Node(
            units=[high, general, low],  # Use unit from dictionary or default
            weights=perm_areas_model[cat_id],
            area=areas.get(cat_id),  # Use predefined area or default
            id=cat_id
        )
        catchments.append(node)  # Store in the list

        # Assign the node as a global variable
        globals()[cat_id] = node


    # Ensure topology only includes nodes that exist in `catchments_ids`
    topology = {
        cat_id: upstream if upstream in catchments_ids else None
        for cat_id, upstream in topology_list.items() if cat_id in catchments_ids
    }

    # Create the Network
    model = Network(
        nodes=catchments,  # Pass list of Node objects
        topology=topology  
    )

    model.reset_states()

    # Set inputs for each node using the manually defined dictionary
    for cat in catchments:
        cat.set_input(inputs[cat.id])  # Correct way to set inputs

    model.set_timestep(1.0)
    model.set_parameters(best_params_dict_model)

    output = model.get_output()

    return output


def generate_nse_results(catchments_ids, daterange, output, observations, quality_masks):


    # Create an empty list to store results
    nse_results_cal = []

    for basin in catchments_ids:
        Qtimeseries = pd.DataFrame(index=daterange)
        Qtimeseries["Qobs"] = observations[basin]
        Qtimeseries["Qcalc"] = output[basin][0]

        hydro_year = calculate_hydro_year(date=Qtimeseries.index, first_month=10)

        nse_value = 1 - obj_fun_nsee(observations=Qtimeseries.iloc[365:, 0].values, 
                                    simulation=Qtimeseries.iloc[365:, 1].values, 
                                    expo=0.5)
                
        bfi_obs = hydroanalysis.streamflow_signatures.calculate_baseflow_index(Qtimeseries.iloc[365:, 0].values, quality_masks[basin][365:], alpha=0.925, num_filters=3, num_reflect=30, returnBF=False)
        bfi_sim = hydroanalysis.streamflow_signatures.calculate_baseflow_index(Qtimeseries.iloc[365:, 1].values, quality_masks[basin][365:], alpha=0.925, num_filters=3, num_reflect=30, returnBF=False)
        
        qmean_obs = hydroanalysis.streamflow_signatures.calculate_q_mean(Qtimeseries.iloc[365:, 0].values, quality_masks[basin][365:])
        qmean_sim = hydroanalysis.streamflow_signatures.calculate_q_mean(Qtimeseries.iloc[365:, 1].values, quality_masks[basin][365:])
        
        try:

            slope_obs = hydroanalysis.streamflow_signatures.calculate_slope_fdc(Qtimeseries.iloc[365:, 0].values, quality_masks[basin][365:])["Sawicz"]
            slope_sim = hydroanalysis.streamflow_signatures.calculate_slope_fdc(Qtimeseries.iloc[365:, 1].values, quality_masks[basin][365:])["Sawicz"]
        except: 
            slope_obs = np.nan
            slope_sim = np.nan
        
        try:
            hfd_obs = hydroanalysis.streamflow_signatures.calculate_hfd_mean(Qtimeseries.iloc[365:, 0].values, quality_masks[basin][365:], hydro_year[365:])["hfd_mean"]
            hfd_sim = hydroanalysis.streamflow_signatures.calculate_hfd_mean(Qtimeseries.iloc[365:, 1].values, quality_masks[basin][365:], hydro_year[365:])["hfd_mean"]

        except:
            hfd_obs = np.nan
            hfd_sim = np.nan

        try:            
            nse_value_bfi = 1 - obj_fun_nsee(observations=hydroanalysis.streamflow_signatures.calculate_baseflow_index(Qtimeseries.iloc[365:, 0].values, quality_masks[basin][365:], alpha=0.925, num_filters=3, num_reflect=30, returnBF=True)[1], 
                                    simulation=hydroanalysis.streamflow_signatures.calculate_baseflow_index(Qtimeseries.iloc[365:, 1].values, quality_masks[basin][365:], alpha=0.925, num_filters=3, num_reflect=30, returnBF=True)[1], 
                                    expo=0.5)
        except:
            nse_value_bfi = np.nan


        nse_results_cal.append({
            "gauge_name": network_estreams.loc[basin, "gauge_name"],
            "basin": basin,
            "nse": nse_value,
            "bfi_obs": bfi_obs,
            "bfi_sim":bfi_sim,
            "nse_value_bfi": nse_value_bfi,
            "qmean_obs": qmean_obs,
            "qmean_sim": qmean_sim,
            "slope_obs": slope_obs,
            "slope_sim": slope_sim,
            "hfd_obs": hfd_obs,
            "hfd_sim": hfd_sim
            })

    # Convert results to DataFrame
    nse_results_df = pd.DataFrame(nse_results_cal)

    return nse_results_df

import re

def is_valid_key(k):
    # Exclude keys that end in Group_X_2
    return not re.search(r'Group_\d+_2$', k)

def is_valid_key_2(k):
    # Include only keys that end in Group_X_2
    return re.search(r'Group_\d+_2$', k)

## Model all time-series using all possible combinations of params

In [12]:
path_inputs = '../data/models/input/subset_2001_2015'

inputs = np.load(path_inputs+'//inputs.npy', allow_pickle=True).item()
observations = np.load(path_inputs+'//observations.npy', allow_pickle=True).item()
areas = np.load(path_inputs+'//areas.npy', allow_pickle=True).item()
perm_areas = np.load(path_inputs+'//perm_areas.npy', allow_pickle=True).item()
perm_areascontinental = np.load(path_inputs+'//perm_areascontinental.npy', allow_pickle=True).item()
perm_areasglobal = np.load(path_inputs+'//perm_areasglobal.npy', allow_pickle=True).item()
quality_masks = np.load(path_inputs+'//quality_masks.npy', allow_pickle=True).item()
rootdepth_mean = np.load(path_inputs+'//rootdepth_mean.npy', allow_pickle=True).item()
waterdeficit_mean= np.load(path_inputs+'//waterdeficit_mean.npy', allow_pickle=True).item()

# Added for random permeabilities:
perm_areas_random = {}

for j in range(1, 8):
    perm_areas_random[f"perm_areas_random{j:02d}"] = np.load(
        os.path.join(path_inputs, f"perm_areas_random{j:02d}.npy"),
        allow_pickle=True
    ).item()

# Filter keys
regional_keys = [k for k in all_param_dicts if "regi" in k and is_valid_key(k)]
continental_keys = [k for k in all_param_dicts if "cont" in k and is_valid_key(k)]
global_keys = [k for k in all_param_dicts if "glob" in k and is_valid_key(k)]
nogeo_keys = [k for k in all_param_dicts if "nogeo" in k and is_valid_key(k)]
random_keys = [k for k in all_param_dicts if "rand" in k and is_valid_key(k)]

output_nogeo_dict = {}
output_regional_dict = {}
output_continental_dict = {}
output_global_dict = {}
output_random_dict = {}

for key in tqdm.tqdm(regional_keys):
        
    catchments_ids = estreams_attributes_clipped_filters[
    (estreams_attributes_clipped_filters.group == key[-7:]) &
    (~estreams_attributes_clipped_filters.index.str.contains("LU"))].index.tolist()

    output = run_model_superflexpy(
        catchments_ids=catchments_ids,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areas
    )
    output_regional_dict[key] = output

for key in tqdm.tqdm(continental_keys):

    catchments_ids = estreams_attributes_clipped_filters[
    (estreams_attributes_clipped_filters.group == key[-7:]) &
    (~estreams_attributes_clipped_filters.index.str.contains("LU"))].index.tolist()
        
    output = run_model_superflexpy(
        catchments_ids=catchments_ids,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areascontinental
    )
    output_continental_dict[key] = output

for key in tqdm.tqdm(global_keys):
    
    catchments_ids = estreams_attributes_clipped_filters[
    (estreams_attributes_clipped_filters.group == key[-7:]) &
    (~estreams_attributes_clipped_filters.index.str.contains("LU"))].index.tolist()
        
    output = run_model_superflexpy(
        catchments_ids=catchments_ids,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areasglobal
    )
    output_global_dict[key] = output


for key in tqdm.tqdm(nogeo_keys):

    catchments_ids = estreams_attributes_clipped_filters[
    (estreams_attributes_clipped_filters.group == key[-7:]) &
    (~estreams_attributes_clipped_filters.index.str.contains("LU"))].index.tolist()
        
    output = run_model_superflexpy_nogeo(
        catchments_ids=catchments_ids,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areas
    )
    output_nogeo_dict[key] = output


for key in tqdm.tqdm(random_keys):

    catchments_ids = estreams_attributes_clipped_filters[
        (estreams_attributes_clipped_filters.group == key[-7:]) &
        (~estreams_attributes_clipped_filters.index.str.contains("LU"))].index.tolist()

    output = run_model_superflexpy_random(
        catchments_ids=catchments_ids,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areas_random[f"perm_areas_random{int(key[-1:]):02d}"]
    )
    output_random_dict[key] = output

100%|██████████| 7/7 [00:55<00:00,  7.88s/it]


In [13]:
path_inputs = '../data/models/input/subset_1988_2001'

inputs = np.load(path_inputs+'//inputs.npy', allow_pickle=True).item()
observations = np.load(path_inputs+'//observations.npy', allow_pickle=True).item()
areas = np.load(path_inputs+'//areas.npy', allow_pickle=True).item()
perm_areas = np.load(path_inputs+'//perm_areas.npy', allow_pickle=True).item()
perm_areascontinental = np.load(path_inputs+'//perm_areascontinental.npy', allow_pickle=True).item()
perm_areasglobal = np.load(path_inputs+'//perm_areasglobal.npy', allow_pickle=True).item()
quality_masks = np.load(path_inputs+'//quality_masks.npy', allow_pickle=True).item()
rootdepth_mean = np.load(path_inputs+'//rootdepth_mean.npy', allow_pickle=True).item()
waterdeficit_mean= np.load(path_inputs+'//waterdeficit_mean.npy', allow_pickle=True).item()

# Added for random permeabilities:
perm_areas_random = {}

for j in range(1, 8):
    perm_areas_random[f"perm_areas_random{j:02d}"] = np.load(
        os.path.join(path_inputs, f"perm_areas_random{j:02d}.npy"),
        allow_pickle=True
    ).item()
 

# Filter keys
regional_keys_2 = [k for k in all_param_dicts if "regi" in k and is_valid_key_2(k)]
continental_keys_2 = [k for k in all_param_dicts if "cont" in k and is_valid_key_2(k)]
global_keys_2 = [k for k in all_param_dicts if "glob" in k and is_valid_key_2(k)]
nogeo_keys_2 = [k for k in all_param_dicts if "nogeo" in k and is_valid_key_2(k)]
random_keys_2 = [k for k in all_param_dicts if "rand" in k and is_valid_key_2(k)]


output_nogeo_dict_8801 = {}
output_regional_dict_8801 = {}
output_continental_dict_8801 = {}
output_global_dict_8801 = {}
output_random_dict_8801 = {}

for key in tqdm.tqdm(regional_keys_2):

    catchments_ids = estreams_attributes_clipped_filters[
    (estreams_attributes_clipped_filters.group == key[-9:-2]) &
    (~estreams_attributes_clipped_filters.index.str.contains("LU"))].index.tolist()
        
    output = run_model_superflexpy(
        catchments_ids=catchments_ids,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areas
    )
    output_regional_dict_8801[key] = output

for key in tqdm.tqdm(continental_keys_2):
    catchments_ids = estreams_attributes_clipped_filters[
    (estreams_attributes_clipped_filters.group == key[-9:-2]) &
    (~estreams_attributes_clipped_filters.index.str.contains("LU"))].index.tolist()
    
    output = run_model_superflexpy(
        catchments_ids=catchments_ids,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areascontinental
    )
    output_continental_dict_8801[key] = output

for key in tqdm.tqdm(global_keys_2):
    catchments_ids = estreams_attributes_clipped_filters[
    (estreams_attributes_clipped_filters.group == key[-9:-2]) &
    (~estreams_attributes_clipped_filters.index.str.contains("LU"))].index.tolist()
        
    output = run_model_superflexpy(
        catchments_ids=catchments_ids,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areasglobal
    )
    output_global_dict_8801[key] = output

for key in tqdm.tqdm(nogeo_keys_2):

    catchments_ids = estreams_attributes_clipped_filters[
    (estreams_attributes_clipped_filters.group == key[-9:-2]) &
    (~estreams_attributes_clipped_filters.index.str.contains("LU"))].index.tolist()
        
    output = run_model_superflexpy_nogeo(
        catchments_ids=catchments_ids,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areas
    )
    
    output_nogeo_dict_8801[key] = output

for key in tqdm.tqdm(random_keys_2):

    catchments_ids = estreams_attributes_clipped_filters[
        (estreams_attributes_clipped_filters.group == key[-9:-2]) &
        (~estreams_attributes_clipped_filters.index.str.contains("LU"))].index.tolist()
    
    print(key)
    print(key[-3:-2])
    print(perm_areas_random[f"perm_areas_random{int(key[-3:-2]):02d}"])
    
    output = run_model_superflexpy_random(
        catchments_ids=catchments_ids,
        best_params_dict_model=all_param_dicts[key],
        perm_areas_model=perm_areas_random[f"perm_areas_random{int(key[-3:-2]):02d}"]
    )
    
    output_random_dict_8801[key] = output

  0%|          | 0/7 [00:00<?, ?it/s]

moselle_best_params_randomcompt_Group_2_2
2
{'BEWA0066': [0.38, 0.43, 0.19], 'BEWA0067': [0.34, 0.27, 0.38], 'BEWA0087': [0.2, 0.37, 0.43], 'BEWA0106': [0.26, 0.26, 0.48], 'BEWA0107': [0.13, 0.27, 0.6], 'BEWA0119': [0.3, 0.47, 0.23], 'DEBU1956': [0.16, 0.15, 0.69], 'DEBU1957': [0.02, 0.44, 0.54], 'DEBU1958': [0.24, 0.19, 0.56], 'DEBU1959': [0.5, 0.28, 0.22], 'DERP2003': [0.22, 0.41, 0.37], 'DERP2004': [0.15, 0.43, 0.42], 'DERP2005': [0.35, 0.07, 0.58], 'DERP2006': [0.19, 0.43, 0.38], 'DERP2007': [0.11, 0.6, 0.29], 'DERP2008': [0.33, 0.15, 0.52], 'DERP2010': [0.16, 0.48, 0.35], 'DERP2011': [0.41, 0.54, 0.05], 'DERP2013': [0.22, 0.43, 0.34], 'DERP2014': [0.23, 0.26, 0.52], 'DERP2015': [0.34, 0.12, 0.54], 'DERP2016': [0.28, 0.5, 0.21], 'DERP2017': [0.47, 0.27, 0.26], 'DERP2018': [0.03, 0.4, 0.57], 'DERP2021': [0.31, 0.29, 0.4], 'DERP2022': [0.09, 0.57, 0.34], 'DERP2023': [0.5, 0.42, 0.08], 'DERP2024': [0.33, 0.48, 0.19], 'DERP2027': [0.23, 0.22, 0.55], 'DERP2028': [0.33, 0.56, 0.11], 'DER

 14%|█▍        | 1/7 [00:05<00:31,  5.18s/it]

moselle_best_params_randomcompt_Group_4_2
4
{'BEWA0066': [0.12, 0.19, 0.68], 'BEWA0067': [0.24, 0.15, 0.61], 'BEWA0087': [0.21, 0.53, 0.26], 'BEWA0106': [0.51, 0.47, 0.02], 'BEWA0107': [0.21, 0.52, 0.28], 'BEWA0119': [0.15, 0.26, 0.59], 'DEBU1956': [0.16, 0.72, 0.12], 'DEBU1957': [0.33, 0.35, 0.32], 'DEBU1958': [0.36, 0.44, 0.2], 'DEBU1959': [0.38, 0.31, 0.31], 'DERP2003': [0.8, 0.16, 0.04], 'DERP2004': [0.3, 0.55, 0.16], 'DERP2005': [0.29, 0.28, 0.43], 'DERP2006': [0.18, 0.4, 0.42], 'DERP2007': [0.29, 0.07, 0.64], 'DERP2008': [0.62, 0.22, 0.16], 'DERP2010': [0.2, 0.52, 0.28], 'DERP2011': [0.12, 0.22, 0.66], 'DERP2013': [0.3, 0.07, 0.63], 'DERP2014': [0.13, 0.51, 0.36], 'DERP2015': [0.35, 0.41, 0.23], 'DERP2016': [0.21, 0.5, 0.29], 'DERP2017': [0.74, 0.0, 0.25], 'DERP2018': [0.57, 0.39, 0.04], 'DERP2021': [0.13, 0.37, 0.5], 'DERP2022': [0.29, 0.52, 0.19], 'DERP2023': [0.28, 0.34, 0.38], 'DERP2024': [0.45, 0.17, 0.38], 'DERP2027': [0.4, 0.58, 0.02], 'DERP2028': [0.39, 0.13, 0.48], 'DERP

 29%|██▊       | 2/7 [00:10<00:26,  5.24s/it]

...

 71%|███████▏  | 5/7 [00:28<00:11,  5.98s/it]

moselle_best_params_randomcompt_Group_7_2
7
{'BEWA0066': [0.11, 0.51, 0.38], 'BEWA0067': [0.31, 0.47, 0.21], 'BEWA0087': [0.54, 0.07, 0.39], 'BEWA0106': [0.26, 0.43, 0.31], 'BEWA0107': [0.51, 0.14, 0.35], 'BEWA0119': [0.12, 0.43, 0.45], 'DEBU1956': [0.12, 0.81, 0.08], 'DEBU1957': [0.5, 0.07, 0.42], 'DEBU1958': [0.29, 0.36, 0.35], 'DEBU1959': [0.16, 0.61, 0.23], 'DERP2003': [0.19, 0.4, 0.41], 'DERP2004': [0.4, 0.22, 0.38], 'DERP2005': [0.29, 0.33, 0.39], 'DERP2006': [0.49, 0.16, 0.34], 'DERP2007': [0.61, 0.03, 0.36], 'DERP2008': [0.37, 0.15, 0.48], 'DERP2010': [0.19, 0.49, 0.32], 'DERP2011': [0.46, 0.06, 0.48], 'DERP2013': [0.39, 0.12, 0.48], 'DERP2014': [0.5, 0.34, 0.16], 'DERP2015': [0.4, 0.42, 0.18], 'DERP2016': [0.4, 0.07, 0.53], 'DERP2017': [0.53, 0.15, 0.32], 'DERP2018': [0.05, 0.37, 0.58], 'DERP2021': [0.21, 0.77, 0.02], 'DERP2022': [0.68, 0.02, 0.3], 'DERP2023': [0.16, 0.44, 0.4], 'DERP2024': [0.78, 0.15, 0.07], 'DERP2027': [0.25, 0.51, 0.25], 'DERP2028': [0.22, 0.4, 0.38], 'DER

 86%|████████▌ | 6/7 [00:34<00:05,  5.87s/it]

moselle_best_params_randomcompt_Group_5_2
5
{'BEWA0066': [0.07, 0.26, 0.67], 'BEWA0067': [0.15, 0.76, 0.09], 'BEWA0087': [0.4, 0.0, 0.6], 'BEWA0106': [0.52, 0.34, 0.15], 'BEWA0107': [0.36, 0.07, 0.57], 'BEWA0119': [0.26, 0.4, 0.34], 'DEBU1956': [0.41, 0.24, 0.35], 'DEBU1957': [0.39, 0.37, 0.25], 'DEBU1958': [0.18, 0.51, 0.31], 'DEBU1959': [0.21, 0.28, 0.51], 'DERP2003': [0.43, 0.26, 0.31], 'DERP2004': [0.27, 0.37, 0.36], 'DERP2005': [0.39, 0.3, 0.31], 'DERP2006': [0.1, 0.38, 0.52], 'DERP2007': [0.66, 0.23, 0.11], 'DERP2008': [0.37, 0.53, 0.1], 'DERP2010': [0.39, 0.05, 0.56], 'DERP2011': [0.08, 0.37, 0.54], 'DERP2013': [0.51, 0.31, 0.18], 'DERP2014': [0.23, 0.09, 0.69], 'DERP2015': [0.32, 0.49, 0.19], 'DERP2016': [0.26, 0.42, 0.32], 'DERP2017': [0.55, 0.43, 0.02], 'DERP2018': [0.81, 0.1, 0.1], 'DERP2021': [0.33, 0.52, 0.15], 'DERP2022': [0.32, 0.18, 0.5], 'DERP2023': [0.37, 0.36, 0.26], 'DERP2024': [0.31, 0.37, 0.32], 'DERP2027': [0.55, 0.28, 0.17], 'DERP2028': [0.43, 0.36, 0.22], 'DERP

100%|██████████| 7/7 [00:39<00:00,  5.64s/it]


In [14]:
# Create the concatenated data for the complete series analysis 
path_inputs = '../data/models/input/subset_1988_2001'
observations1 = np.load(path_inputs + '//observations.npy', allow_pickle=True).item()
quality_masks1 = np.load(path_inputs + '//quality_masks.npy', allow_pickle=True).item()

path_inputs = '../data/models/input/subset_2001_2015'
observations2 = np.load(path_inputs + '//observations.npy', allow_pickle=True).item()
quality_masks2 = np.load(path_inputs + '//quality_masks.npy', allow_pickle=True).item()

observations_cal = {}
quality_masks_cal = {}

for key in observations1.keys():
    arr1 = np.atleast_1d(observations1[key])
    arr2 = np.atleast_1d(observations2.get(key, np.array([])))

    # Remove the first 365 days from the second dataset
    arr2_trimmed = arr2[365:] if arr2.size > 365 else np.array([])

    observations_cal[key] = np.concatenate([arr1, arr2_trimmed])

for key in quality_masks1.keys():
    arr1 = np.atleast_1d(quality_masks1[key])
    arr2 = np.atleast_1d(quality_masks2.get(key, np.array([])))

    # Remove the first 365 days from the second dataset
    arr2_trimmed = arr2[365:] if arr2.size > 365 else np.array([])

    quality_masks_cal[key] = np.concatenate([arr1, arr2_trimmed])

In [15]:
# Load both input files
path_inputs_1 = '../data/models/input/subset_1988_2001/inputs.npy'
path_inputs_2 = '../data/models/input/subset_2001_2015/inputs.npy'

inputs1 = np.load(path_inputs_1, allow_pickle=True).item()
inputs2 = np.load(path_inputs_2, allow_pickle=True).item()

# Initialize new dictionaries
precipitation_cal = {}
temperature_cal = {}
evaporation_cal = {}

for key in inputs1.keys():
    # Get (P, T, PET) tuples from each period
    p1, t1, pet1 = map(np.atleast_1d, inputs1[key])
    p2, t2, pet2 = map(np.atleast_1d, inputs2.get(key, ([], [], [])))

    # Concatenate and assign
    precipitation_cal[key] = np.concatenate([p1, p2])
    temperature_cal[key] = np.concatenate([t1, t2])
    evaporation_cal[key] = np.concatenate([pet1, pet2])

In [16]:
output_global_dict_cal = {}

for param_key in output_global_dict:
    param_key_8801 = param_key + "_2"

    if param_key_8801 in output_global_dict_8801:
        merged_outputs = {}

        for gauge_id in output_global_dict[param_key]:
            if gauge_id in output_global_dict_8801[param_key_8801]:
                # Ensure both arrays are flat (1D)
                series_8801 = np.ravel(output_global_dict_8801[param_key_8801][gauge_id])
                series_recent = np.ravel(output_global_dict[param_key][gauge_id])
                #concatenated = np.concatenate([series_8801, series_recent])
                
                # Remove first 365 days from recent series
                series_recent_trimmed = series_recent[365:] if series_recent.size > 365 else np.array([])
                concatenated = np.concatenate([series_8801, series_recent_trimmed])

                merged_outputs[gauge_id] = [concatenated]

        output_global_dict_cal[param_key] = merged_outputs

In [17]:
output_continental_dict_cal = {}

for param_key in output_continental_dict:
    param_key_8801 = param_key + "_2"

    if param_key_8801 in output_continental_dict_8801:
        merged_outputs = {}

        for gauge_id in output_continental_dict[param_key]:
            if gauge_id in output_continental_dict_8801[param_key_8801]:
                # Ensure both arrays are flat (1D)
                series_8801 = np.ravel(output_continental_dict_8801[param_key_8801][gauge_id])
                series_recent = np.ravel(output_continental_dict[param_key][gauge_id])
                #concatenated = np.concatenate([series_8801, series_recent])
                
                # Remove first 365 days from recent series
                series_recent_trimmed = series_recent[365:] if series_recent.size > 365 else np.array([])
                concatenated = np.concatenate([series_8801, series_recent_trimmed])

                merged_outputs[gauge_id] = [concatenated]

        output_continental_dict_cal[param_key] = merged_outputs

In [18]:
output_regional_dict_cal = {}

for param_key in output_regional_dict:
    param_key_8801 = param_key + "_2"

    if param_key_8801 in output_regional_dict_8801:
        merged_outputs = {}

        for gauge_id in output_regional_dict[param_key]:
            if gauge_id in output_regional_dict_8801[param_key_8801]:
                # Ensure both arrays are flat (1D)
                series_8801 = np.ravel(output_regional_dict_8801[param_key_8801][gauge_id])
                series_recent = np.ravel(output_regional_dict[param_key][gauge_id])
                #concatenated = np.concatenate([series_8801, series_recent])

                # Remove first 365 days from recent series
                series_recent_trimmed = series_recent[365:] if series_recent.size > 365 else np.array([])
                concatenated = np.concatenate([series_8801, series_recent_trimmed])

                merged_outputs[gauge_id] = [concatenated]

        output_regional_dict_cal[param_key] = merged_outputs

In [19]:
output_nogeo_dict_cal = {}

for param_key in output_nogeo_dict:
    param_key_8801 = param_key + "_2"

    if param_key_8801 in output_nogeo_dict_8801:
        merged_outputs = {}

        for gauge_id in output_nogeo_dict[param_key]:
            if gauge_id in output_nogeo_dict_8801[param_key_8801]:
                # Ensure both arrays are flat (1D)
                series_8801 = np.ravel(output_nogeo_dict_8801[param_key_8801][gauge_id])
                series_recent = np.ravel(output_nogeo_dict[param_key][gauge_id])
                #concatenated = np.concatenate([series_8801, series_recent])

                # Remove first 365 days from recent series
                series_recent_trimmed = series_recent[365:] if series_recent.size > 365 else np.array([])
                concatenated = np.concatenate([series_8801, series_recent_trimmed])

                merged_outputs[gauge_id] = [concatenated]

        output_nogeo_dict_cal[param_key] = merged_outputs

In [20]:
output_random_dict_cal = {}

for param_key in output_random_dict:
    param_key_8801 = param_key + "_2"

    if param_key_8801 in output_random_dict_8801:
        merged_outputs = {}

        for gauge_id in output_random_dict[param_key]:
            if gauge_id in output_random_dict_8801[param_key_8801]:
                # Ensure both arrays are flat (1D)
                series_8801 = np.ravel(output_random_dict_8801[param_key_8801][gauge_id])
                series_recent = np.ravel(output_random_dict[param_key][gauge_id])
                #concatenated = np.concatenate([series_8801, series_recent])

                # Remove first 365 days from recent series
                series_recent_trimmed = series_recent[365:] if series_recent.size > 365 else np.array([])
                concatenated = np.concatenate([series_8801, series_recent_trimmed])

                merged_outputs[gauge_id] = [concatenated]

        output_random_dict_cal[param_key] = merged_outputs

## Save the time-series in netcdfs

In [21]:
import xarray as xr
import numpy as np

# Adjust these according to your data
group_suffixes = ["Group_1",
                  "Group_2",
                  "Group_3",
                  "Group_4", "Group_5", 
                  "Group_6", 
                  "Group_7"
                  ]

gauge_ids = list(observations_cal.keys())  # Your observations dict should be preloaded
time_index = pd.date_range(start="1988-10-01", end="2015-09-30", freq='D')

In [22]:
# Build datasets
datasets = {}

for suffix in group_suffixes:
    reg_key = f"moselle_best_params_regicompt_{suffix}"
    cont_key = f"moselle_best_params_contcompt_{suffix}"
    glob_key = f"moselle_best_params_globcompt_{suffix}"
    nogeo_key = f"moselle_best_params_nogeot_{suffix}"
    random_key = f"moselle_best_params_randomcompt_{suffix}"

    reg_data = []
    cont_data = []
    glob_data = []
    nogeo_data = []
    random_data = []

    group_gauge_ids = []

    for gauge in gauge_ids:
        if gauge in output_regional_dict_cal[reg_key] and \
           gauge in output_continental_dict_cal[cont_key] and \
           gauge in output_nogeo_dict_cal[nogeo_key] and \
            gauge in output_random_dict_cal[random_key] and \
           gauge in output_global_dict_cal[glob_key]:

            reg_data.append(output_regional_dict_cal[reg_key][gauge][0])
            cont_data.append(output_continental_dict_cal[cont_key][gauge][0])
            glob_data.append(output_global_dict_cal[glob_key][gauge][0])
            nogeo_data.append(output_nogeo_dict_cal[nogeo_key][gauge][0])
            random_data.append(output_random_dict_cal[random_key][gauge][0])

            group_gauge_ids.append(gauge)

    if group_gauge_ids:
        ds = xr.Dataset(
            data_vars={
                "regional": (["gauge_id", "date"], reg_data),
                "continental": (["gauge_id", "date"], cont_data),
                "global": (["gauge_id", "date"], glob_data),
                "nogeo": (["gauge_id", "date"], nogeo_data),
                "random": (["gauge_id", "date"], random_data)

            },
            coords={
                "gauge_id": group_gauge_ids,
                "date": time_index
            }
        )
        datasets[suffix] = ds

# Save each group to a separate NetCDF file using scipy (no need for netCDF4)
for suffix, ds in datasets.items():
    ds.to_netcdf(rf"../results/sim/calibration/simu_cal_{suffix}.nc", engine="scipy")

# End